In [42]:
# Data handling
import pandas as pd
import numpy as np

# Progress bars
from tqdm import tqdm

# Hugging Face Transformers (model, tokenizer, pipelines)
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainingArguments,
    EarlyStoppingCallback,
    PegasusTokenizer,
)

# Metrics and Evaluation
from rouge_score import rouge_scorer
from sklearn.metrics import average_precision_score

# Torch for GPU
import torch

# For displaying results in notebook
from IPython.display import display

# Plotting and Visualization (optional but useful)
import matplotlib.pyplot as plt
import seaborn as sns

# Logging
import logging

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [44]:
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

print("Train set:")
display(train_df.head(2))
print("Validation set:")
display(val_df.head(2))
print("Test set:")
display(test_df.head(2))

Train set:


,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...


Validation set:


,id,article,highlights
0,61df4979ac5fcc2b71be46ed6fe5a46ce7f071c3,"Sally Forrest, an actress-dancer who graced th...","Sally Forrest, an actress-dancer who graced th..."
1,21c0bd69b7e7df285c3d1b1cf56d4da925980a68,A middle-school teacher in China has inked hun...,Works include pictures of Presidential Palace ...


Test set:


,id,article,highlights
0,92c514c913c0bdfe25341af9fd72b29db544099b,Ever noticed how plane seats appear to be gett...,Experts question if packed out planes are put...
1,2003841c7dc0e7c5b1a248f9cd536d727f27a45a,A drunk teenage boy had to be rescued by secur...,Drunk teenage boy climbed into lion enclosure ...


In [4]:
print("Train columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

# Check for missing values
print("Missing values in train:")
print(train_df.isnull().sum())
print("Missing values in validation:")
print(val_df.isnull().sum())
print("Missing values in test:")
print(test_df.isnull().sum())

Train columns: ['id', 'article', 'highlights']
Validation columns: ['id', 'article', 'highlights']
Test columns: ['id', 'article', 'highlights']
Missing values in train:
id            0
article       0
highlights    0
dtype: int64
Missing values in validation:
id            0
article       0
highlights    0
dtype: int64
Missing values in test:
id            0
article       0
highlights    0
dtype: int64


In [5]:
# Remove missing values if any
train_df = train_df.dropna(subset=['article', 'highlights'])
val_df = val_df.dropna(subset=['article', 'highlights'])
test_df = test_df.dropna(subset=['article', 'highlights'])

# Strip whitespace
train_df['article'] = train_df['article'].str.strip()
train_df['highlights'] = train_df['highlights'].str.strip()
val_df['article'] = val_df['article'].str.strip()
val_df['highlights'] = val_df['highlights'].str.strip()
test_df['article'] = test_df['article'].str.strip()
test_df['highlights'] = test_df['highlights'].str.strip()

In [6]:
print("Average article length (train):", train_df['article'].str.len().mean())
print("Average summary length (train):", train_df['highlights'].str.len().mean())

Average article length (train): 4033.6608652342456
Average summary length (train): 294.7703900554834


In [7]:
# Models to benchmark (feel free to swap with other variants if needed)
model_infos = [
    {"name": "BART", "hf_id": "facebook/bart-large-cnn"},
    {"name": "T5", "hf_id": "t5-base"},
    {"name": "Pegasus", "hf_id": "google/pegasus-cnn_dailymail"},
    {"name": "FLAN-T5", "hf_id": "google/flan-t5-base"},
    {"name": "BigBird-Pegasus", "hf_id": "google/bigbird-pegasus-large-arxiv"},  # Or use the cnn_dailymail variant
]

# Function to load model and tokenizer
def load_model(model_hf_id):
    if "pegasus" in model_hf_id:
        # Force slow tokenizer for Pegasus
        tokenizer = PegasusTokenizer.from_pretrained(model_hf_id, use_fast=False)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_hf_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_hf_id).to(device)
    model.eval()
    return model, tokenizer


In [8]:
def generate_summary(model, tokenizer, article, 
                     max_input_length=1024, max_output_length=128):
    # Tokenize and truncate long articles as needed
    inputs = tokenizer(
        article, 
        return_tensors="pt",
        truncation=True, 
        max_length=max_input_length,
        padding="max_length"
    ).to(device)
    with torch.no_grad():
        summary_ids = model.generate(
            **inputs, 
            max_length=max_output_length,
            min_length=30,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [16]:
# Use a subset for demo/experimentation (increase to more samples for final runs)
N_SAMPLES = 100
sample_df = test_df.sample(N_SAMPLES, random_state=42).reset_index(drop=True)

all_results = []

for m_info in model_infos:
    print(f"Evaluating: {m_info['name']}")
    model, tokenizer = load_model(m_info['hf_id'])
    gens = []
    for article in tqdm(sample_df['article'], desc=f"Summarizing ({m_info['name']})"):
        summary = generate_summary(model, tokenizer, article)
        gens.append(summary)
    # Compute ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = [
        scorer.score(ref, pred)
        for ref, pred in zip(sample_df['highlights'], gens)
    ]
    # Aggregate ROUGE scores
    rouge1 = np.mean([s['rouge1'].fmeasure for s in scores])
    rouge2 = np.mean([s['rouge2'].fmeasure for s in scores])
    rougel = np.mean([s['rougeL'].fmeasure for s in scores])
    all_results.append({
        "Model": m_info['name'],
        "ROUGE-1": rouge1,
        "ROUGE-2": rouge2,
        "ROUGE-L": rougel,
    })
    # Save outputs for later qualitative inspection
    sample_df[f'{m_info["name"]}_gen'] = gens

# Final results as DataFrame
results_df = pd.DataFrame(all_results)
display(results_df)

Evaluating: BART


Summarizing (BART): 100%|██████████| 100/100 [02:32<00:00,  1.53s/it]


Evaluating: T5


Summarizing (T5): 100%|██████████| 100/100 [03:52<00:00,  2.32s/it]


Evaluating: Pegasus


...

pytorch_model.bin:   0%|          | 0.00/2.31G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.31G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]


Summarizing (BigBird-Pegasus): 100%|██████████| 100/100 [06:10<00:00,  3.70s/it]


,Model,ROUGE-1,ROUGE-2,ROUGE-L
0,BART,0.440294,0.200576,0.299882
1,T5,0.393009,0.166178,0.271495
2,Pegasus,0.413114,0.207893,0.301254
3,FLAN-T5,0.389524,0.160537,0.262596
4,BigBird-Pegasus,0.180260,0.024893,0.116558


In [19]:
from sklearn.metrics import precision_score

def calc_precision(reference, prediction):
    ref_tokens = set(reference.lower().split())
    pred_tokens = set(prediction.lower().split())
    tp = len(ref_tokens & pred_tokens)
    fp = len(pred_tokens - ref_tokens)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    return precision

# Example for all models
precision_results = []

for m_info in model_infos:
    gen_col = f"{m_info['name']}_gen"
    precisions = [
        calc_precision(ref, pred)
        for ref, pred in zip(sample_df['highlights'], sample_df[gen_col])
    ]
    precision_results.append({
        "Model": m_info["name"],
        "Token Precision": np.mean(precisions)
    })

precision_df = pd.DataFrame(precision_results)
display(precision_df)

,Model,Token Precision
0,BART,0.379945
1,T5,0.387582
2,Pegasus,0.476856
3,FLAN-T5,0.338748
4,BigBird-Pegasus,0.177515


In [21]:
!pip install bert-score

from bert_score import score as bert_score

# For a single model:
P, R, F1 = bert_score(
    sample_df['BART_gen'].tolist(),    # preds
    sample_df['highlights'].tolist(),  # refs
    lang='en', rescale_with_baseline=True
)
print(f"BART BERTScore F1: {F1.mean():.4f}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

C:\Users\Adidya\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Adidya\M_H_Project\cache\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BART BERTScore F1: 0.2822


FINE TUNED CODE: